# Multi-Step Evaluation with Tree of Thoughts (ToT)

This notebook evaluates the `gemini-3.1-flash-lite` model on Killer Sudoku puzzles 1-5 (6x6) using a Multi-step Prompt format (analysis and fill loop) enhanced with Tree of Thoughts (ToT) DFS backtracking search.

In [11]:
import os
import json
import time
import sys
from pathlib import Path
from google import genai
from google.genai import types

# Setup Google GenAI Client
os.environ["GOOGLE_CLOUD_PROJECT"] = "project-038ccd57-3d62-4aac-8b5"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

client = genai.Client()
model_LLM = "gemini-3.1-flash-lite"

In [12]:
# Dynamic path resolution to ensure it runs from any context
cwd = Path.cwd()
if (cwd / 'tot_solver.py').exists():
    tot_dir = cwd
elif (cwd / 'ToT' / 'tot_solver.py').exists():
    tot_dir = cwd / 'ToT'
elif (cwd / 'cs106' / 'ToT' / 'tot_solver.py').exists():
    tot_dir = cwd / 'cs106' / 'ToT'
else:
    raise FileNotFoundError('Cannot locate cs106/ToT resources from current directory.')

sys.path.insert(0, str(tot_dir.resolve()))
from tot_solver import run_multi_step_tot_search

cs106_dir = tot_dir.parent
dataset_dir = cs106_dir / 'dataset'
output_dir = tot_dir / 'outputs' / 'multi-prompt'
output_dir.mkdir(parents=True, exist_ok=True)

print(f"ToT directory: {tot_dir}")
print(f"Dataset directory: {dataset_dir}")
print(f"Output directory: {output_dir}")

ToT directory: d:\Study\HK6\CS106-AI\DoAn\CS106-Sudoku-Bench\cs106\ToT
Dataset directory: d:\Study\HK6\CS106-AI\DoAn\CS106-Sudoku-Bench\cs106\dataset
Output directory: d:\Study\HK6\CS106-AI\DoAn\CS106-Sudoku-Bench\cs106\ToT\outputs\multi-prompt


In [13]:
# --- 6x6 Prompts ---
analysis_prompt_6x6 = """
You are an expert Sudoku reasoning engine. Your current task is strictly limited to ANALYSIS and NOTE-TAKING. 
Do NOT attempt to solve the puzzle completely. Do NOT output a final grid. Do NOT make eager guesses.

Your objective is to carefully scan the current board state, apply standard Sudoku rules along with the specific variant constraints, and generate a logical scratchpad. This scratchpad will be used in the next step to confidently commit digits.

[KNOWLEDGE RECAP]
1. The Grid & Core Rules:
- The grid is 6x6, containing 6 rows and 6 columns.
- Notation: "r1c1" indicates the cell at row 1 and column 1.
- The grid consists of 6 rectangular 2x3 subgrids (2 rows tall, 3 columns wide).
- Standard Sudoku Rule: Each row, each column, and each 2x3 subgrid must contain the numbers 1 through 6 exactly once.
- Killer Sudoku Rule: The grid is divided into "Cages" (contiguous groups of cells). The sum of the numbers in each cage must perfectly match its provided target sum.
- Non-Repeating Rule: Numbers cannot repeat within a single cage.

2. Mathematical Facts & Solving Strategies:
- Sum to 21: Since the numbers 1 through 6 add up to 21, every completely filled row, column, and 2x3 subgrid must sum exactly to 21.
- A 6-cell cage must also have a target sum of 21.
- Deduction Tip: For any subset of cells (row, column, or subgrid), if all but one cell are filled, the final cell's value must be 21 minus the sum of the known cells.

3. Reference Material: The Cheat-Sheet
Important Interpretation Rules for the cheat-sheet:
- Combinations are written as sequences of individual digits without separators.
- Each digit in the sequence represents one distinct number in the combination.
- Example: "12" means the combination {{1, 2}}. "135" means the combination {{1, 3, 5}}.
Here is your cheat-sheet for cage combinations:
{cheat_sheet}

[LAST MOVE LOG]
- Cell Modified: {last_cell} (Value: {last_value})
- AI's Reasoning: "{last_reasoning}"
- Confidence: {last_is_certain}

[CURRENT STATE]
Current board:
{current_board}

Cages:
{cages}

[PREVIOUS ANALYSIS]
{previous_analysis}

INSTRUCTIONS:
1. Scan the board for the most constrained areas (cells, rows, columns, or variant shapes with the fewest empty spaces).
2. Eliminate candidate digits for these empty cells based on the active rules.
3. If a cell is reduced to a single valid digit, explicitly highlight it in the "Deductions" section with irrefutable logical proof.
4. Output your analysis EXACTLY matching the Markdown template below. Do not add conversational filler.

{analysis_template}
"""

fill_prompt_6x6 = """
You are the Execution Engine of an advanced Sudoku solver. Your task is to evaluate the provided logical analysis and commit EXACTLY ONE valid digit to the board.

[KNOWLEDGE RECAP]
1. The Grid & Core Rules:
- The grid is 6x6, containing 6 rows and 6 columns.
- Notation: "r1c1" indicates the cell at row 1 and column 1.
- The grid consists of 6 rectangular 2x3 subgrids (2 rows tall, 3 columns wide).
- Standard Sudoku Rule: Each row, each column, and each 2x3 subgrid must contain the numbers 1 through 6 exactly once.
- Killer Sudoku Rule: The grid is divided into "Cages" (contiguous groups of cells). The sum of the numbers in each cage must perfectly match its provided target sum.
- Non-Repeating Rule: Numbers cannot repeat within a single cage.

2. Mathematical Facts & Solving Strategies:
- Sum to 21: Since the numbers 1 through 6 add up to 21, every completely filled row, column, and 2x3 subgrid must sum exactly to 21.
- A 6-cell cage must also have a target sum of 21.
- Deduction Tip: For any subset of cells (row, column, or subgrid), if all but one cell are filled, the final cell's value must be 21 minus the sum of the known cells.

3. Reference Material: The Cheat-Sheet
Important Interpretation Rules for the cheat-sheet:
- Combinations are written as sequences of individual digits without separators.
- Each digit in the sequence represents one distinct number in the combination.
- Example: "12" means the combination {{1, 2}}. "135" means the combination {{1, 3, 5}}.
Here is your cheat-sheet for cage combinations:
{cheat_sheet}

[CURRENT STATE]
Current board:
{current_board}

Cages:
{cages}

[LOGICAL ANALYSIS]
{current_analysis}

INSTRUCTIONS:
1. Review the "3. Deductions & Bottlenecks" section of the Logical Analysis carefully.
2. Identify ONE cell where a digit is mathematically forced. You can ONLY modify cells that currently contain 0.
3. If there is NO forced cell, review the "2. Candidate Elimination" section and make your best educated guess for the cell with the fewest candidates.
4. You MUST output your decision for exactly ONE cell. Abstaining is strictly prohibited.
5. Output your response STRICTLY as a valid JSON object.
"""

# --- 9x9 Prompts ---
analysis_prompt_9x9 = """
You are an expert Killer Sudoku reasoning engine.

Your current task is strictly limited to ANALYSIS and NOTE-TAKING.
Do NOT solve the puzzle completely.
Do NOT output a final grid.
Do NOT make guesses.

Your objective is to analyze the current board state as a constraint system and create a logical scratchpad.
This scratchpad will be used in the next step to commit only mathematically justified digits.

[KNOWLEDGE RECAP]

1. Grid & Core Rules

- The grid size is {grid_size}x{grid_size}.
- Notation: "r1c1" means row 1, column 1.
- Each cell must contain one digit from 1 to {grid_size}.
- Subgrid shape: {box_shape}.
- Standard Sudoku Rule:
  - Each row must contain digits 1 to {grid_size} exactly once.
  - Each column must contain digits 1 to {grid_size} exactly once.
  - Each subgrid must contain digits 1 to {grid_size} exactly once.
- Killer Sudoku Rule:
  - The grid is divided into cages.
  - The digits in each cage must sum to the cage target.
  - Digits cannot repeat within the same cage.
- Important uniqueness rule:
  - Different cells are NOT required to have different values globally.
  - Two cells must be different only if they share a row, a column, a 3x3 box, or the same killer cage.
  - If algebra implies cell A = cell B, this is NOT a contradiction unless A and B are constrained to be different by row, column, box, or cage.
  - Do not reject a deduction only because two cells in different constraints may have the same value.
2. Equation System View

Treat the puzzle as a constraint system.

- Number of row equations: {grid_size}
- Number of column equations: {grid_size}
- Number of subgrid equations: {grid_size}
- Number of cage equations: {cage_count}
- Total sum equations: {total_equations}

Every completed row, column, and subgrid sums to {unit_sum}.

Each equation has two constraints:
1. The values must sum to the target.
2. The values inside the same row, column, subgrid, or cage must be distinct.

3. Reference Material: Cheat Sheet

Important interpretation rules:

- Combinations are written as digit sequences without separators.
- Example: "12" means {{1, 2}}.
- Example: "135" means {{1, 3, 5}}.
- Use the cheat sheet to restrict cage combinations.

Cheat sheet:

{cheat_sheet}

[LAST MOVE LOG]

- Cell Modified: {last_cell}
- Value: {last_value}
- AI's Reasoning: "{last_reasoning}"
- Confidence: {last_is_certain}

[CURRENT STATE]

Current board:

{current_board}

Empty cells you are allowed to reason/fill:

{empty_cells}

IMPORTANT:
- You may ONLY propose cells from this empty_cells list.
- Any cell not in this list is already filled.
- Never output a filled cell as a forced placement.

Cages:

{cages}

[PREVIOUS ANALYSIS]

{previous_analysis}

[INSTRUCTIONS]

1. First, inspect the LAST MOVE LOG.
2. If the last move is not N/A, update all equations affected by that move:
   - its row equation
   - its column equation
   - its 3x3 box equation
   - its cage equation
3. For each affected equation, compute:
   - known values
   - remaining cells
   - remaining sum
   - allowed digits
4. Propagate the updated restrictions to related rows, columns, boxes, and cages.
5. Use the cheat sheet to restrict cage combinations.
6. List candidate eliminations only when they are logically justified.
7. Highlight forced placements only if they are mathematically certain.
8. If no forced placement exists, clearly state NO_CERTAIN_MOVE.
9. Keep the analysis compact. Focus only on affected rows, columns, boxes, cages, and the strongest deductions.
10. Do not enumerate every candidate for every empty cell.
11. Output your analysis exactly following the Markdown template below.

{analysis_template}
"""

fill_prompt_9x9 = """
You are the Execution Engine of an advanced Killer Sudoku solver.

Your task is to evaluate the provided logical analysis and commit at most ONE valid digit to the board.

Do NOT guess.
Only fill a digit if it is mathematically forced.

[KNOWLEDGE RECAP]

1. Grid & Core Rules

- The grid size is {grid_size}x{grid_size}.
- Notation: "r1c1" means row 1, column 1.
- Each cell must contain one digit from 1 to {grid_size}.
- Subgrid shape: {box_shape}.
- Standard Sudoku Rule:
  - Each row must contain digits 1 to {grid_size} exactly once.
  - Each column must contain digits 1 to {grid_size} exactly once.
  - Each subgrid must contain digits 1 to {grid_size} exactly once.
- Killer Sudoku Rule:
  - The digits in each cage must sum to the cage target.
  - Digits cannot repeat within the same cage.
- Important uniqueness rule:
  - Different cells are NOT required to have different values globally.
  - Two cells must be different only if they share a row, a column, a 3x3 box, or the same killer cage.
  - If algebra implies cell A = cell B, this is NOT a contradiction unless A and B are constrained to be different by row, column, box, or cage.
  - Do not reject a deduction only because two cells in different constraints may have the same value.

2. Equation System View

- Every completed row, column, and subgrid sums to {unit_sum}.
- Number of cage equations: {cage_count}
- Total equations: {total_equations}

3. Reference Material: Cheat Sheet

{cheat_sheet}

[CURRENT STATE]

Current board:

{current_board}

Empty cells you are allowed to reason/fill:

{empty_cells}

IMPORTANT:
- You may ONLY propose cells from this empty_cells list.
- Any cell not in this list is already filled.
- Never output a filled cell as a forced placement.

Cages:

{cages}

[LOGICAL ANALYSIS]

{current_analysis}

[INSTRUCTIONS]

1. Review the analysis carefully.
2. Choose ONE cell only if its value is mathematically forced.
3. You can ONLY modify cells that currently contain 0.
4. Before choosing a cell, verify from the CURRENT BOARD that the cell contains 0.
5. Never choose a cell that already has a non-zero value.
6. If the analysis suggests filling a non-zero cell, reject that suggestion and return no placement.
7. If no forced cell exists, return no placement.
8. Do NOT make educated guesses.
9. Output strictly as a valid JSON object.
"""

In [14]:
puzzle_ids = list(range(7, 8))
all_results = []

for pid in puzzle_ids:
    p_file = dataset_dir / f"puzzle_{pid:02d}.json"
    print(f"\nEvaluating Puzzle {pid:02d} ({p_file.name})...")
    
    with open(p_file, 'r', encoding='utf-8') as f:
        puzzle = json.load(f)
        
    grid_size = puzzle['grid_size']
    
    # Select prompts, cheat sheet, and templates
    if grid_size == 6:
        analysis_prompt = analysis_prompt_6x6
        fill_prompt = fill_prompt_6x6
        cheat_sheet_file = cs106_dir / "6x6" / "killer_sudoku_cheat_sheet.md"
        template_file = cs106_dir / "6x6" / "analysis_template.md"
    else:
        analysis_prompt = analysis_prompt_9x9
        fill_prompt = fill_prompt_9x9
        cheat_sheet_file = cs106_dir / "9x9" / "killer_sudoku_cheat_sheet_9x9.md"
        template_file = cs106_dir / "9x9" / "analysis_template_9x9.md"
        
    with open(cheat_sheet_file, 'r', encoding='utf-8') as f:
        cheat_sheet = f.read()
    puzzle['cheat_sheet'] = cheat_sheet

    with open(template_file, 'r', encoding='utf-8') as f:
        analysis_template = f.read()

    # Form cages text
    cages_text = []
    for cage in puzzle['cages']:
        cell_strs = [f"r{r+1}c{c+1}" for r, c in cage['cells']]
        cages_text.append("- " + " + ".join(cell_strs) + f" = {cage['sum']}")

    start_time = time.time()
    
    # Run multi-step ToT search solver
    solved_board, execution_log, status = run_multi_step_tot_search(
        client=client,
        model=model_LLM,
        board=puzzle['puzzle'],
        puzzle=puzzle,
        analysis_prompt=analysis_prompt,
        fill_prompt=fill_prompt,
        cages_text=cages_text,
        analysis_template=analysis_template,
        use_solution_validation=True,
        max_depth=50,
        k=3,
        temperature=0.7
    )
    
    elapsed = time.time() - start_time
    
    log_data = {
        "puzzle_id": puzzle['id'],
        "difficulty": puzzle['difficulty'],
        "grid_size": grid_size,
        "model": model_LLM,
        "status": status,
        "time_seconds": elapsed,
        "total_steps": len([e for e in execution_log if "chosen_cell" in e]),
        "prediction": solved_board,
        "solution": puzzle['solution'],
        "execution_log": execution_log
    }
    
    out_file = output_dir / f"log_puzzle_{pid:02d}.json"
    with open(out_file, 'w', encoding='utf-8') as f:
        json.dump(log_data, f, ensure_ascii=False, indent=2)
        
    print(f"  Result: {status} | Steps: {log_data['total_steps']} | Time: {elapsed:.2f}s | Saved to: {out_file.name}")
    all_results.append(log_data)
    
    time.sleep(2) # respect rate limits


Evaluating Puzzle 07 (puzzle_07.json)...
  Result: Failed | Steps: 0 | Time: 39.79s | Saved to: log_puzzle_07.json


## 3. Summary of runs

In [15]:
print("\nEvaluation Complete!")
print("-" * 40)
for r in all_results:
    print(f"Puzzle {r['puzzle_id']:02d} ({r['grid_size']}x{r['grid_size']}) | {r['status']} | Steps: {r['total_steps']} | Time: {r['time_seconds']:.2f}s")


Evaluation Complete!
----------------------------------------
Puzzle 07 (9x9) | Failed | Steps: 0 | Time: 39.79s
